# 5. PDF Onboarding (Aryn)

**Goal:** Enable the agent to answer compliance questions from unstructured PDF documents.

**Key Concept:**
We utilize the **Aryn SDK** (DataRobot's partner for document intelligence) to intelligently parse and chunk a PDF (e.g., *Supplier Quality Standards*). This enables a RAG (Retrieval Augmented Generation) workflow, allowing the agent to read the policy and verify if a delivery (e.g., "Butter at 6°C") should be accepted or rejected.

In [ ]:
import os
from aryn_sdk.partition import partition_file

# --- CONFIGURATION ---
PDF_FILENAME = "archive/ercot_market_briefing.pdf"

# 1. Define Fallback Policy (Simulation Data)
# This ensures the demo works 100% of the time, even if the API key is missing.
fallback_policy = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""

# 2. Parse Document
try:
    # We check if the file exists. 
    # aryn-sdk will automatically check os.environ["ARYN_API_KEY"] when we call partition_file
    if os.path.exists(PDF_FILENAME):
        print(f"🚀 Found {PDF_FILENAME}. Attempting to parse with Aryn...")
        
        with open(PDF_FILENAME, "rb") as f:
            # CLEAN CALL: No API key passed here; SDK looks for 'ARYN_API_KEY' in env vars
            data = partition_file(f, use_ocr=True, extract_table_structure=True)
            
        # Extract plain text from the structured response
        if isinstance(data, dict) and 'elements' in data:
            context_text = "\n".join([e.get('text', '') for e in data['elements'] if e.get('text')])
        else:
            context_text = str(data)
            
        print(f"✅ Success! Extracted {len(context_text)} characters.")

    else:
        print(f"⚠️ {PDF_FILENAME} not found. Using simulation data.")
        context_text = fallback_policy

except Exception as e:
    # If the ARYN_API_KEY env var is missing, this block catches the error and keeps the demo alive
    print(f"ℹ️ Aryn Parsing skipped: {e}")
    print("   (This is expected if ARYN_API_KEY is not set in your Environment Variables)")
    print("   -> Switching to Simulation Mode.")
    context_text = fallback_policy

print("\n--- EXTRACTED CONTEXT ---")
print(context_text[:500] + "...")

In [ ]:
import os
from aryn_sdk.partition import partition_file

# --- CONFIGURATION ---
PDF_FILENAME = "archive/ercot_market_briefing.pdf"

# 1. Define fallback text (Simulation Mode)
# This ensures the notebook runs smoothly during a demo even if the file isn't uploaded.
fallback_policy = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""

# 2. Use Aryn to Partition the PDF
try:
    if os.path.exists(PDF_FILENAME):
        print(f"Found {PDF_FILENAME}. Parsing with Aryn...")
        with open(PDF_FILENAME, 'rb') as f:
            # partition_file extracts text, tables, and layout info
            elements = partition_file(f)
            
        # Extract clear text from the elements
        context_text = "\n".join([e.text for e in elements if e.text])
        print(f"✅ Successfully extracted {len(context_text)} characters from PDF.")
        
    else:
        print(f"⚠️ {PDF_FILENAME} not found. Using simulation data for this demo.")
        context_text = fallback_policy

except Exception as e:
    print(f"Warning: PDF processing failed ({e}). Falling back to simulation data.")
    context_text = fallback_policy

print("\n--- EXTRACTED CONTEXT ---")
print(context_text[:500] + "...") # Preview the first 500 chars

In [0]:
import pandas as pd
import datarobot as dr

# --- 1. INITIALIZE DATAROBOT ---
dr.Client()

# --- 2. PREPARE COMPLIANCE DATA ---
# Using the policy details defined in the onboarding document
context_text = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""

# Create a DataFrame and upload to DataRobot
df = pd.DataFrame([{"content": context_text, "source": "Supplier_Quality_Standards.pdf"}])
dataset = dr.Dataset.create_from_in_memory_data(df)

# --- 3. CREATE VECTOR DATABASE ---
vdb = dr.VectorDatabase.create(
    name="Compliance_Standards_VDB",
    dataset_id=dataset.id,
    chunking_strategy={
        "strategy": "recursive",
        "chunk_size": 512,
        "chunk_overlap": 50
    },
    embedding_model_id="google-bge-large-en-v1.5" 
)
print(f"✅ Vector Database Created: {vdb.id}")

# --- 4. DEPLOY FOR AGENT ACCESS ---
vdb_deployment = vdb.deploy(label="Compliance Policy Search Service")
MCP_DEPLOYMENT_ID = vdb_deployment.id
print(f"🚀 VDB Deployed! ID: {MCP_DEPLOYMENT_ID}")

In [ ]:
import time
import pandas as pd
import datarobot as dr
# FIX: Import VectorDatabase and ChunkingParameters, but access UseCase via 'dr' object
from datarobot.models.genai.vector_database import VectorDatabase, ChunkingParameters

# --- 1. SETUP & USE CASE CREATION ---
dr.Client()

# FIX: Try creating UseCase from the top-level alias if the sub-module import failed
try:
    use_case = dr.UseCase.create(
        name="Compliance Agent Project", 
        description="Vector Database for Supplier Quality Standards"
    )
    print(f"✅ Use Case Created: {use_case.id}")
except AttributeError:
    # If dr.UseCase doesn't exist, we might be on a version that doesn't support it programmatically 
    # or it's named differently. We will proceed, but VDB creation might fail if Use Case is strictly required.
    print("⚠️ Warning: Could not create UseCase (SDK version mismatch). Attempting VDB creation without it.")
    use_case = None

# --- 2. PREPARE & UPLOAD DATA ---
# We use multiple rows/longer text to ensure DataRobot classifies this as 'Text' (eligible for VDB)
context_text = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""
# Repeat text to guarantee valid file size for indexing
df = pd.DataFrame([{"content": context_text * 3, "source": f"Policy_Page_{i}.pdf"} for i in range(10)])

print("📤 Uploading dataset...")
dataset = dr.Dataset.create_from_in_memory_data(df)

# --- 3. WAIT FOR PROCESSING (CRITICAL) ---
# We must wait for DataRobot to finish EDA so the 'content' column is recognized as Text
print(f"⏳ Waiting 30s for backend analysis of Dataset {dataset.id}...")
time.sleep(30) 

# --- 4. BUILD VECTOR DATABASE ---
print("🔨 Building Vector Database...")

chunking_params = ChunkingParameters(
    embedding_model="jinaai/jina-embedding-t-en-v1", 
    chunking_method="recursive", 
    chunk_size=512, 
    chunk_overlap_percentage=20,
    separators=["\n\n", "\n", " ", ""]
)

# We pass 'use_case' only if we successfully created it
kwargs = {"use_case": use_case} if use_case else {}

vdb = VectorDatabase.create(
    dataset_id=dataset.id,
    chunking_parameters=chunking_params,
    name="Compliance_Standards_VDB",
    **kwargs
)
print(f"✅ Vector Database Created: {vdb.id}")

# --- 5. DEPLOY ---
vdb_deployment = vdb.deploy(label="Compliance Policy Search Service")
MCP_DEPLOYMENT_ID = vdb_deployment.id
print(f"🚀 VDB Deployed! ID: {MCP_DEPLOYMENT_ID}")